# GSM8K Step 1: Paper-Aligned Judging

This notebook qualifies FullKV with the eight-shot Chain-of-Thought prompt from ChunkKV Appendix G, Table 30, verifies exact token parity through the unpruned custom-cache generation path, and only then enables TDC-KV compression. Run cells in order. Do not use Run All.

## 1. Clone or update `branch-h`

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO = Path("/content/Tier-based-KV-cache-with-Dependency-Aware-Chunk-Scoring")
URL = "https://github.com/JayGor-13/Tier-based-KV-cache-with-Dependency-Aware-Chunk-Scoring.git"

if not REPO.exists():
    subprocess.run(
        ["git", "clone", "--branch", "branch-h", "--single-branch", URL, str(REPO)],
        check=True,
    )
else:
    subprocess.run(["git", "pull", "--ff-only", "origin", "branch-h"], cwd=REPO, check=True)

os.chdir(REPO)
REVISION = subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], text=True).strip()
print("Repository:", REPO)
print("Revision:", REVISION)

## 2. Install and verify the environment

In [ ]:
packages = [
    "transformers>=4.43,<6",
    "datasets>=5.0.1",
    "accelerate>=1.14.0",
    "pandas>=2.2",
    "pytest>=8.4",
]
subprocess.run([sys.executable, "-m", "pip", "install", "-q", *packages], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".", "--no-deps"], check=True)
print("Installation complete.")

In [ ]:
import torch
import transformers

assert torch.cuda.is_available(), "Attach a Colab GPU before continuing."
print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("GPU:", torch.cuda.get_device_name(0))
print("VRAM (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1))

In [ ]:
test_result = subprocess.run(
    [
        sys.executable, "-m", "pytest", "-q",
        "tests/test_gsm8k_protocol.py",
        "tests/test_eval_metrics.py",
        "tests/test_hf_runner.py",
        "tests/test_hf_cache_e2e.py",
    ],
    cwd=REPO,
    text=True,
    timeout=1200,
)
print("Return code:", test_result.returncode)
if test_result.returncode != 0:
    raise RuntimeError("GSM8K protocol or cache parity tests failed.")

## 3. Define the guarded GSM8K runner

In [ ]:
import json
import time
from datetime import datetime, timezone

import pandas as pd
from IPython.display import display

RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
OUTPUT_DIR = REPO / "outputs" / "gsm8k_step1" / RUN_ID
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
SMOKE_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
PAPER_MODEL = "Qwen/Qwen2-7B-Instruct"
PROTOCOL = "chunkkv_gsm8k_8shot"
DATASET = (
    "name=gsm8k_chunkkv,source=openai/gsm8k,adapter=gsm8k,"
    "protocol=chunkkv_gsm8k_8shot,config=main,split=test,"
    "prompt_field=question,answer_field=answer"
)
print("Output directory:", OUTPUT_DIR)

def judgment_frame(payload):
    rows = []
    for run in payload.get("runs", []):
        if run.get("status") != "ok":
            continue
        judgment = run.get("judgment") or {}
        rows.append({
            "sample_id": run.get("sample_id"),
            "method": run.get("method"),
            "budget_type": run.get("config", {}).get("budget_type"),
            "budget_value": run.get("config", {}).get("budget_value"),
            "prediction_answer": judgment.get("normalized_prediction"),
            "gold_answer": judgment.get("normalized_gold"),
            "correct": judgment.get("correct"),
            "prompt_sha256": run.get("prompt_sha256"),
        })
    return pd.DataFrame(rows)

def validate_gsm8k_payload(payload, *, require_parity=False):
    assert payload["protocols"]["gsm8k_chunkkv"]["protocol"] == PROTOCOL
    errors = [run for run in payload["runs"] if run.get("status") != "ok"]
    if errors:
        raise AssertionError(f"{len(errors)} run(s) failed: {errors[0]}")
    for run in payload["runs"]:
        assert run.get("protocol") == PROTOCOL
        assert run.get("prompt_sha256")
        assert run.get("input_token_sha256")
        assert isinstance(run.get("generated_token_ids"), list)
        assert run.get("judgment", {}).get("judge") == "gsm8k_final_numeric_exact_match"
    if require_parity:
        parity = payload["summary"]["fullkv_parity"]
        assert parity["count"] > 0
        assert parity["all_passed"] is True, parity
    return payload

def run_gsm8k(*, label, model, methods, budget_ratios="", max_samples=10, run_parity=False, timeout_minutes=90):
    output = OUTPUT_DIR / f"{label}.json"
    command = [
        sys.executable, "-u", "scripts/run_hf_grid.py",
        "--models", model,
        "--datasets", DATASET,
        "--methods", methods,
        "--budget-ratios", budget_ratios,
        "--thetas", "0.3",
        "--recent-windows", "16",
        "--alphas", "0.6",
        "--dependency-top-k", "8",
        "--max-chunk-tokens", "64",
        "--min-budget-utilization", "0.99",
        "--max-budget-shortfall-tokens", "1",
        "--prefill-block-size", "128",
        "--tier1-score-mode", "dependency",
        "--max-samples", str(max_samples),
        "--max-length", "2048",
        "--max-new-tokens", "256",
        "--device", "cuda",
        "--dtype", "float16",
        "--allow-level2-fallback",
        "--progress",
        "--output", str(output),
    ]
    if run_parity:
        command.append("--require-fullkv-parity")
    print("Command:", " ".join(command))
    started = time.perf_counter()
    try:
        completed = subprocess.run(command, cwd=REPO, text=True, timeout=timeout_minutes * 60)
    except subprocess.TimeoutExpired as exc:
        raise RuntimeError(f"{label} exceeded {timeout_minutes} minutes.") from exc
    print(f"Elapsed: {(time.perf_counter() - started) / 60:.1f} minutes")
    if not output.exists():
        raise RuntimeError(f"{label} did not produce {output}.")
    payload = validate_gsm8k_payload(
        json.loads(output.read_text(encoding="utf-8")),
        require_parity=run_parity,
    )
    if completed.returncode != 0:
        raise RuntimeError(f"{label} failed with return code {completed.returncode}.")
    display(judgment_frame(payload))
    return payload

## 4. T4 engineering smoke and generation-path parity

This uses the small 1.5B model to verify execution and exact generation parity. Its task score is diagnostic only and must not be treated as a paper-quality FullKV baseline. Compression remains disabled.

In [ ]:
smoke = run_gsm8k(
    label="engineering_parity_10",
    model=SMOKE_MODEL,
    methods="fullkv",
    max_samples=10,
    run_parity=True,
    timeout_minutes=90,
)

In [ ]:
baseline = smoke["summary"]["baseline_qa_summary"]
parity = smoke["summary"]["fullkv_parity"]
diagnostics = []
for run in smoke["runs"]:
    if run.get("status") != "ok" or run.get("method") != "fullkv":
        continue
    judgment = run.get("judgment") or {}
    diagnostics.append({
        "sample_id": run.get("sample_id"),
        "prediction_answer": judgment.get("normalized_prediction"),
        "gold_answer": judgment.get("normalized_gold"),
        "correct": judgment.get("correct"),
        "generated_tokens": len(run.get("generated_token_ids", [])),
        "prediction": run.get("prediction", ""),
    })
display(pd.DataFrame(diagnostics))
print("Smoke-model metric:", baseline["primary_metric"])
print("Smoke-model score (diagnostic only):", baseline["primary_score"])
print("Token parity:", parity["token_parity_rate"])
print("Text parity:", parity["text_parity_rate"])
assert baseline["primary_metric"] == "gsm8k_accuracy"
assert parity["all_passed"] is True
print("Engineering parity gate: PASSED")
if baseline["primary_score"] == 0.0:
    print("The 1.5B smoke model is not a valid quality baseline. Continue with the paper-model qualification on an L4 or A100.")

## 5. Paper-model FullKV qualification

ChunkKV reports Qwen2-7B-Instruct on GSM8K. Use an L4 (24 GB) or A100 runtime for this stage; a T4 is appropriate only for the engineering smoke above. The gate requires a nontrivial FullKV baseline before any compression result is accepted.

In [ ]:
gpu_memory_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
if gpu_memory_gb < 20:
    raise RuntimeError(
        f"{torch.cuda.get_device_name(0)} has {gpu_memory_gb:.1f} GB VRAM. "
        "Switch the Colab runtime to an L4 or A100 before loading Qwen2-7B-Instruct."
    )
print("Paper-model runtime check: PASSED")

In [ ]:
qualification = run_gsm8k(
    label="qwen2_7b_fullkv_qualification_20",
    model=PAPER_MODEL,
    methods="fullkv",
    max_samples=20,
    run_parity=True,
    timeout_minutes=180,
)

In [ ]:
baseline = qualification["summary"]["baseline_qa_summary"]
parity = qualification["summary"]["fullkv_parity"]
print("Paper-model FullKV metric:", baseline["primary_metric"])
print("Paper-model FullKV score:", baseline["primary_score"])
print("Token parity:", parity["token_parity_rate"])
assert baseline["primary_metric"] == "gsm8k_accuracy"
assert baseline["primary_score"] >= 0.30, (
    "Paper-model FullKV score is unexpectedly low; inspect its predictions before compression."
)
assert parity["all_passed"] is True
print("Paper-model qualification gate: PASSED")

## 6. Twenty-sample compression pilot

Run only after the qualification gate passes. This is a development curve, not the final paper-scale table.

In [ ]:
compression_pilot = run_gsm8k(
    label="qwen2_7b_tdc_compression_20",
    model=PAPER_MODEL,
    methods="fullkv,tdc_kv",
    budget_ratios="0.75,0.5,0.25",
    max_samples=20,
    run_parity=False,
    timeout_minutes=240,
)

In [ ]:
summary_rows = []
for group in compression_pilot["grouped_results"]:
    qa = group["qa_summary"]
    cache = group["cache_summary"]
    summary_rows.append({
        "method": group["method"],
        "budget_type": group["budget"]["type"],
        "budget_value": group["budget"]["value"],
        "samples": group["run_summary"]["successful"],
        "gsm8k_accuracy": qa["primary_score"],
        "retention": cache["avg_retention_ratio"],
        "compression_multiplier": cache["avg_compression_multiplier"],
        "budget_gap": cache["avg_budget_gap"],
    })
summary_frame = pd.DataFrame(summary_rows).sort_values(
    ["method", "budget_value"], na_position="first"
)
display(summary_frame)

## 7. Archive the evidence

In [ ]:
import shutil

archive = shutil.make_archive(str(OUTPUT_DIR), "zip", root_dir=OUTPUT_DIR)
print("Archive:", archive)